# VGG16 Cosine Verification
Pipeline:
1. Load celebrity dataset
2. Extract embeddings using VGG16
3. Verify using cosine similarity threshold = 0.6
4. Report Accuracy, Precision, Recall, F1, FNMR, FMR

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# install libraries
%pip install tqdm -q

In [3]:
# unzip celebrity dataset from drive
import zipfile
print("Unzipping celebrity_dataset...")
with zipfile.ZipFile('/content/drive/MyDrive/celebrity_dataset.zip', 'r') as z:
    z.extractall('/content/')
print("Done!")

Unzipping celebrity_dataset...
Done!


In [4]:
# check structure
import os
print("Gallery identities:", len(os.listdir('/content/celebrity_dataset/gallery')))
print("Probe identities:",   len(os.listdir('/content/celebrity_dataset/probe')))
print("Sample:", os.listdir('/content/celebrity_dataset/gallery')[:3])

Gallery identities: 100
Probe identities: 100
Sample: ['Hugo_Weaving', 'Hal_Holbrook', 'Sally_Field']


In [5]:
# imports
import os
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

In [6]:
# paths
GALLERY_DIR       = '/content/celebrity_dataset/gallery'
PROBE_DIR         = '/content/celebrity_dataset/probe'
GALLERY_EMBED_DIR = '/content/celebrity_dataset/vgg_gallery_embeddings'
PROBE_EMBED_DIR   = '/content/celebrity_dataset/vgg_probe_embeddings'
DRIVE_SAVE        = '/content/drive/MyDrive'

for folder in [GALLERY_EMBED_DIR, PROBE_EMBED_DIR]:
    os.makedirs(folder, exist_ok=True)

THRESHOLD = 0.6
print("Paths ready! Threshold =", THRESHOLD)

Paths ready! Threshold = 0.6


In [7]:
# load VGG16 pretrained
print("Loading VGG16...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

vgg_model = models.vgg16(pretrained=True)
# remove final classification layer - use 4096 features
vgg_model.classifier = torch.nn.Sequential(
    *list(vgg_model.classifier.children())[:-1]
)
vgg_model.eval()
vgg_model = vgg_model.to(device)
print("VGG16 Loaded!")

Loading VGG16...
Device: cuda


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:06<00:00, 86.1MB/s]


VGG16 Loaded!


In [8]:
# VGG transform and embedding function
vgg_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

def get_vgg_embedding(img_path):
    try:
        img = Image.open(img_path).convert('RGB')
        tensor = vgg_transform(img).unsqueeze(0).to(device)
        with torch.no_grad():
            embedding = vgg_model(tensor).squeeze().cpu().numpy()
        # L2 normalise
        norm = np.linalg.norm(embedding)
        if norm > 0:
            embedding = embedding / norm
        return embedding
    except Exception as e:
        print(f"Error: {img_path} - {e}")
        return None

print("VGG embedding function ready!")

VGG embedding function ready!


In [9]:
# extract gallery embeddings
print("Extracting gallery embeddings...")
gallery_saved = 0

for identity in tqdm(os.listdir(GALLERY_DIR)):
    identity_dir = os.path.join(GALLERY_DIR, identity)
    if not os.path.isdir(identity_dir):
        continue

    save_dir = os.path.join(GALLERY_EMBED_DIR, identity)
    os.makedirs(save_dir, exist_ok=True)

    for img_file in os.listdir(identity_dir):
        if not img_file.lower().endswith(('.jpg', '.jpeg', '.png', '.webp')):
            continue

        img_path = os.path.join(identity_dir, img_file)
        embedding = get_vgg_embedding(img_path)

        if embedding is not None:
            base_name = os.path.splitext(img_file)[0]
            np.save(os.path.join(save_dir, base_name + '.npy'), embedding)
            gallery_saved += 1

print(f"Gallery embeddings saved: {gallery_saved}")

Extracting gallery embeddings...


100%|██████████| 100/100 [00:02<00:00, 43.65it/s]

Gallery embeddings saved: 100


In [10]:
# extract probe embeddings
print("Extracting probe embeddings...")
probe_saved = 0

for identity in tqdm(os.listdir(PROBE_DIR)):
    identity_dir = os.path.join(PROBE_DIR, identity)
    if not os.path.isdir(identity_dir):
        continue

    save_dir = os.path.join(PROBE_EMBED_DIR, identity)
    os.makedirs(save_dir, exist_ok=True)

    for img_file in os.listdir(identity_dir):
        if not img_file.lower().endswith(('.jpg', '.jpeg', '.png', '.webp')):
            continue

        img_path = os.path.join(identity_dir, img_file)
        embedding = get_vgg_embedding(img_path)

        if embedding is not None:
            base_name = os.path.splitext(img_file)[0]
            np.save(os.path.join(save_dir, base_name + '.npy'), embedding)
            probe_saved += 1

print(f"Probe embeddings saved: {probe_saved}")

Extracting probe embeddings...


100%|██████████| 100/100 [00:01<00:00, 91.17it/s]

Probe embeddings saved: 100


In [11]:
# load gallery database
def load_gallery_db(gallery_embed_root):
    gallery_db = {}
    total = 0
    for identity in os.listdir(gallery_embed_root):
        identity_dir = os.path.join(gallery_embed_root, identity)
        if not os.path.isdir(identity_dir):
            continue
        gallery_db[identity] = []
        for emb_file in os.listdir(identity_dir):
            if not emb_file.endswith('.npy'):
                continue
            emb = np.load(os.path.join(identity_dir, emb_file))
            gallery_db[identity].append(emb)
            total += 1
    print(f"Gallery loaded: {len(gallery_db)} identities, {total} embeddings")
    return gallery_db

gallery_db = load_gallery_db(GALLERY_EMBED_DIR)

Gallery loaded: 100 identities, 100 embeddings


In [12]:
# cosine verification
results = []

for identity in tqdm(os.listdir(PROBE_EMBED_DIR)):
    probe_identity_dir = os.path.join(PROBE_EMBED_DIR, identity)
    if not os.path.isdir(probe_identity_dir):
        continue

    for emb_file in os.listdir(probe_identity_dir):
        if not emb_file.endswith('.npy'):
            continue

        probe_emb = np.load(os.path.join(probe_identity_dir, emb_file))

        # compare against all gallery identities
        identity_scores = {}
        for gal_identity, gal_embeddings in gallery_db.items():
            sims = [
                cosine_similarity(
                    probe_emb.reshape(1, -1),
                    g.reshape(1, -1)
                )[0][0]
                for g in gal_embeddings
            ]
            identity_scores[gal_identity] = max(sims)

        sorted_scores = sorted(
            identity_scores.items(),
            key=lambda x: x[1],
            reverse=True
        )
        best_identity   = sorted_scores[0][0]
        best_similarity = sorted_scores[0][1]

        decision = 1 if best_similarity >= THRESHOLD else 0
        label    = 1 if (identity in gallery_db and best_identity == identity) else 0

        results.append({
            'true_identity':   identity,
            'predicted':       best_identity,
            'best_similarity': best_similarity,
            'decision':        decision,
            'label':           label,
        })

print(f"Total probes tested: {len(results)}")

100%|██████████| 100/100 [00:03<00:00, 26.50it/s]

Total probes tested: 100


In [13]:
# compute metrics
df = pd.DataFrame(results)

y_true = df['label']
y_pred = df['decision']

cm = confusion_matrix(y_true, y_pred)
TN, FP, FN, TP = cm.ravel()

accuracy  = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, zero_division=0)
recall    = recall_score(y_true, y_pred, zero_division=0)
f1        = f1_score(y_true, y_pred, zero_division=0)
FNMR = FN / (TP + FN) if (TP + FN) > 0 else 0
FMR  = FP / (TN + FP) if (TN + FP) > 0 else 0
TAR  = TP / (TP + FN) if (TP + FN) > 0 else 0
TRR  = TN / (TN + FP) if (TN + FP) > 0 else 0

print("====== VGG16 + Cosine Threshold=0.6 ======")
print(f"Accuracy:              {accuracy:.4f}  ({accuracy*100:.2f}%)")
print(f"Precision:             {precision:.4f}")
print(f"Recall (genuine acc):  {recall:.4f}")
print(f"F1-score:              {f1:.4f}")
print(f"FNMR (genuine degrad): {FNMR:.4f}")
print(f"FMR  (impostor):       {FMR:.4f}")
print()
print("====== CONFUSION MATRIX ======")
print(f"TP: {TP}  TN: {TN}  FP: {FP}  FN: {FN}")
print()
print("====== ADDITIONAL METRICS ======")
print(f"TAR: {TAR:.4f}")
print(f"TRR: {TRR:.4f}")

====== VGG16 + Cosine Threshold=0.6 ======
Accuracy:              0.1400  (14.00%)
Precision:             0.0227
Recall (genuine acc):  1.0000
F1-score:              0.0444
FNMR (genuine degrad): 0.0000
FMR  (impostor):       0.8776

====== CONFUSION MATRIX ======
TP: 2  TN: 12  FP: 86  FN: 0

====== ADDITIONAL METRICS ======
TAR: 1.0000
TRR: 0.1224


In [14]:
# save results to drive
df.to_csv(f'{DRIVE_SAVE}/vggnet_cosine_results.csv', index=False)
print("Results CSV saved!")

with open(f'{DRIVE_SAVE}/vggnet_cosine_metrics.txt', 'w') as f:
    f.write("VGG16 + Cosine Threshold=0.6\n")
    f.write("==============================\n")
    f.write(f"Accuracy:              {accuracy:.4f}\n")
    f.write(f"Precision:             {precision:.4f}\n")
    f.write(f"Recall (genuine acc):  {recall:.4f}\n")
    f.write(f"F1-score:              {f1:.4f}\n")
    f.write(f"FNMR (genuine degrad): {FNMR:.4f}\n")
    f.write(f"FMR  (impostor):       {FMR:.4f}\n")
    f.write(f"TP: {TP}  TN: {TN}  FP: {FP}  FN: {FN}\n")

print("Metrics TXT saved!")
print("All results saved to Drive!")

Results CSV saved!
Metrics TXT saved!
All results saved to Drive!
